← [05 · BLS vs TLS](05_bls_vs_tls.ipynb) · [Index](README.md) · [06 · Designing a search](06_search_design.ipynb) →
<!--nav-->

# Reproducing a known result: recovering Kepler-8 b from raw photometry

**Goal:** independently re-derive the orbital period and planet/star radius ratio of a *known* transiting exoplanet, straight from Kepler light curves, and check our answers against the published values.

This is a deliberate warm-up. Before hunting for anything *new*, we want to prove the pipeline works on a signal whose right answer we already know. Recovering a known planet builds the validation instinct we'll need when a candidate is genuinely novel and there's no answer key.

**Target: Kepler-8 b** (`KIC 6922244`) — a hot Jupiter with a deep, unambiguous transit.

| Quantity | Published value |
|---|---|
| Orbital period | 3.52254 d |
| Rp/Rs (radius ratio) | ~0.094 |

Pipeline: `search → download → stitch → flatten → BLS periodogram → recover period → fold → measure depth → compare to literature`.

In [ ]:
import matplotlib.pyplot as plt

from skyplay import data, detrend, periods, plotting, vetting

plotting.use_style()

target = data.TARGETS['kepler-8']
print(target)
print(target.note)
print(f'\npublished period : {target.period} d')
print(f'published Rp/Rs  : {target.rp_rs}')

## 1. Find the data

Kepler observed in ~3-month "quarters". Each quarter is a separate data product; we search for all of the long-cadence (30-min) light curves for this star.

In [ ]:
search = data.search('kepler-8')      # wraps lk.search_lightcurve
print(f'{len(search)} light-curve products found')
search

## 2. Download and stitch

We take a handful of quarters (enough for a clean detection, few enough to download quickly), then stitch them into one continuous light curve. Each quarter is normalized before stitching so quarter-to-quarter flux offsets don't create artificial jumps.

`data.load_stitched` does the search, download, normalize, concatenate and NaN-drop in one call, and caches the result to `data/cache/` so re-running this notebook doesn't re-download anything.

> Pass `quarters=tuple(range(1, 18))` to use every quarter — it makes the folded transit crisper at the cost of a longer first download. Note that stitching normalizes each quarter separately, so any real variability on timescales longer than a quarter is removed along with the instrumental step.

In [ ]:
lc = data.load_stitched('kepler-8', quarters=(1, 2, 3, 4))
print(f'{len(lc)} cadences over {(lc.time.max() - lc.time.min()).value:.0f} days')
lc.scatter(s=1)
plt.show();

## 3. Flatten

The raw curve has slow trends (stellar variability, instrument drift) that we need to divide out before searching. `detrend.savgol_flatten` wraps lightkurve's `flatten()`, a Savitzky–Golay filter.

The window must be **longer than the transit** or the filter will treat the transit itself as trend and iron it flat. We use 18.8 days here — far longer than the ~2.4 h transit, so we're safe.

📐 Note the units: lightkurve's own `flatten(window_length=...)` counts **cadences**, and the `901` you'll see in older code is 901 × 30 min ≈ 18.8 days. `skyplay.detrend` takes **days** so the number means the same thing whichever filter you use. Notebook 03 section 4 measures what happens when this window is too short — the answer is more interesting than it looks.

In [ ]:
flat, trend = detrend.savgol_flatten(lc, window_days=18.8)
flat.scatter(s=1)
plt.show();

## 4. Search for a period (Box Least Squares)

BLS slides a box-shaped dip across the data at thousands of trial periods and reports where a periodic dip best fits. The peak of the periodogram is our **independently recovered period** — we are *not* telling the algorithm the answer, only a plausible range (1–10 days).

In [ ]:
bls = periods.bls_search(flat, period_min=1, period_max=10, n_periods=20_000)

print(bls.summary())

# P/2 and 2P are flagged: an eclipsing binary found at half its true period puts a
# strong peak at those aliases, which is what the odd-even test in notebook 04 catches.
plotting.plot_spectrum(bls)
plt.show();

### ✅ Validation check 1 — period

In [ ]:
print(bls.compare_to(target.period))

## 5. Fold on the recovered period

If the period is right, folding every cycle on top of each other should stack all the transits into a single clean dip at phase 0. This is the visual proof the signal is real and periodic.

In [ ]:
plotting.plot_folded(flat.time.value, flat.flux.value, bls.period, bls.epoch,
                     phase_window=0.05)
plt.show();

## 6. Measure the transit depth → Rp/Rs

For a transit, depth ≈ (Rp/Rs)² — the fraction of starlight blocked is the ratio of the planet's disk area to the star's. So `sqrt(depth)` gives us the planet-to-star radius ratio, a second independent quantity to check.

We measure depth empirically: median flux out-of-transit minus median flux in-transit.

> **Gotcha worth internalizing:** a tempting earlier step is `remove_outliers()` to tidy the curve. Its default clips *low* outliers too — and a transit **is** a cluster of low outliers. Run it before this step and you'll shave the bottoms off your transits and *underestimate the depth by ~10×*. The period would still survive (enough points remain), but a physical measurement would be quietly wrong. Preprocessing that looks harmless can eat your signal; that's exactly the kind of self-inflicted error validation is meant to catch.

In [ ]:
# The in-transit window is the recovered half-duration, expressed as a phase fraction.
halfwidth = (bls.duration / 2) / bls.period

report = vetting.vet(flat.time.value, flat.flux.value, bls.period, bls.epoch,
                     halfwidth=halfwidth)

print(f'in-transit phase half-width : {halfwidth:.4f}')
print(f'transit depth               : {report.transit_depth * 1e6:.0f} ppm')
print(f'implied Rp/Rs               : {report.rp_rs:.4f}')

### ✅ Validation check 2 — radius ratio

In [ ]:
err = abs(report.rp_rs - target.rp_rs) / target.rp_rs * 100
print(f'recovered Rp/Rs : {report.rp_rs:.4f}')
print(f'published Rp/Rs : {target.rp_rs:.4f}')
print(f'agreement       : {err:.1f}% off')

## 7. And the vetting checks, for free

We measured the depth with `vetting.vet`, which also ran the notebook 04 checks along the
way. Worth printing: a capstone that recovers the right numbers but skips vetting teaches
the wrong instinct, because a mis-folded eclipsing binary would also produce a confident
period and a clean-looking fold.

In [ ]:
print(report.summary())

## What we just did

Starting from nothing but raw pixel-derived light curves, we independently recovered **both** the orbital period (to ~0.005%) and the planet/star radius ratio (to 0.6%) of Kepler-8 b, and confirmed them against the literature. Two independent quantities agreeing with published values is strong evidence the pipeline is sound.

**Why this matters for discovery:** when we later point BLS + folding at a *candidate* with no answer key, we'll trust it precisely because it passed this test on a known object. We also learned a concrete failure mode — low-outlier clipping silently eating the signal — that we now know to watch for.

This notebook now runs on `skyplay`, so the same five calls work on any target: `load_stitched` → `savgol_flatten` → `bls_search` → `plot_folded` → `vet`.

**Natural next steps:**
- Re-run on a *different* known planet (e.g. a shallower, longer-period one) to see where the pipeline gets harder.
- Cross-check the epoch `t0` and duration against the literature too.
- Swap `bls_search` for `tls_search` here and see whether the sharper peak from notebook 05 tightens the recovered period against the published value.
- Then graduate to a *bounded novelty search* — but characterise your sensitivity first.

**Next:** [`06_search_design.ipynb`](06_search_design.ipynb) — scoping a real search, and measuring what your pipeline can and cannot detect. A search that finds nothing has told you nothing until you know what it was capable of finding.